In [1]:
# ============================================================
# Cell 1 — LLM Adapter (Async + FairLLM Compatible)
# ============================================================

import os
import json
import asyncio
import httpx
from openai import AsyncOpenAI

# ==============================
# Configure your OpenAI API
# ==============================
# Paste your OpenAI key here
OPENAI_API_KEY = "notakey"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# ==============================
# Build Async OpenAI Client
# ==============================
# Disable HTTP/2 for stability on Windows / proxy networks
openai_http_client = httpx.AsyncClient(
    http2=False,
    timeout=httpx.Timeout(20.0, connect=20.0, read=20.0, write=20.0),
    trust_env=True,
    headers={"User-Agent": "fairllm-agent-weather/1.0"}
)

# Create AsyncOpenAI client
aclient = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    http_client=openai_http_client,
    timeout=140.0,
)

# ============================================
# Define Async Adapter for FairLLM Planner
# ============================================
class OpenAIChatAdapter:
    """
    Async OpenAI adapter that works with FairLLM SimpleReActPlanner.
    Exposes the `ainvoke(messages)` method expected by the planner.
    """
    def __init__(self, model_name: str = "gpt-4.1-mini", temperature: float = 0.2):
        self.model_name = model_name
        self.temperature = temperature

    async def ainvoke(self, messages):
        # Convert FairLLM messages → OpenAI format
        oa_messages = [
            {
                "role": (m.get("role") if isinstance(m, dict) else getattr(m, "role", "user")),
                "content": (m.get("content") if isinstance(m, dict) else getattr(m, "content", "")),
            }
            for m in messages
        ]

        resp = await aclient.chat.completions.create(
            model=self.model_name,
            messages=oa_messages,
            temperature=self.temperature,
        )
        content = (resp.choices[0].message.content or "").strip()
        return type("LLMMessage", (), {"content": content})  # Planner expects .content

    def invoke(self, prompt: str) -> str:
        """Optional sync helper (not used by planner)."""
        async def _go():
            msg = await self.ainvoke([{"role": "user", "content": prompt}])
            return msg.content
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(_go())
        else:
            return loop.create_task(_go())

# ==============================
# Initialize and Test LLM
# ==============================
llm = OpenAIChatAdapter()

async def _ping():
    r = await llm.ainvoke([{"role": "user", "content": "Say OK"}])
    print("LLM ping:", r.content)

try:
    loop = asyncio.get_running_loop()
    loop.create_task(_ping())
except RuntimeError:
    asyncio.run(_ping())


LLM ping: OK


In [2]:
# Cell 2 — Tools: geocoding, weather, drone profile, location search, and weather limits

import json
import time
import httpx
from typing import Any, Dict, List, Optional


class GeocodeCityTool:
    name = "geocode_city"
    description = "Geocode a city name to latitude and longitude using Open-Meteo Geocoding."
    input_schema = {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }

    def _parse_input(self, input_obj):
        # Accept raw string (city) or dict or JSON-string
        if isinstance(input_obj, str):
            s = input_obj.strip()

            # First, try to JSON-decode the string.
            # This handles cases like '"Denver"' or '{"city": "Denver"}'.
            try:
                decoded = json.loads(s)
                # If we got a dict with city, use that
                if isinstance(decoded, dict) and "city" in decoded:
                    return str(decoded["city"]).strip()
                # If we got a plain string, treat that as the city name
                if isinstance(decoded, str):
                    return decoded.strip()
            except Exception:
                # Not valid JSON, fall back to previous behavior
                pass

            # Old behavior: JSON object style string
            if s.startswith("{") and s.endswith("}"):
                try:
                    obj = json.loads(s)
                    return str(obj.get("city") or "").strip()
                except Exception:
                    pass

            # Otherwise just treat it as a bare city name
            return s

        if isinstance(input_obj, dict):
            city = input_obj.get("city")
            if city:
                return str(city).strip()

        raise ValueError("location_search input must be a city name string or {'city': '...'} dict.")

    def use(self, input_obj):
        city = self._parse_input(input_obj)
        params = {"name": city, "count": 1}
        with httpx.Client(timeout=10.0, headers={"User-Agent": "fairllm-agent-weather/1.0"}, http2=False) as cli:
            last_status = None
            for _ in range(3):
                r = cli.get("https://geocoding-api.open-meteo.com/v1/search", params=params)
                last_status = r.status_code
                if last_status == 200:
                    data = r.json()
                    results = data.get("results") or []
                    if results:
                        c0 = results[0]
                        out = {
                            "city": c0.get("name"),
                            "country": c0.get("country"),
                            "lat": c0.get("latitude"),
                            "lon": c0.get("longitude"),
                        }
                        return json.dumps(out)
                    return json.dumps({"error": f"City not found: {city}"})
                time.sleep(0.6)
        return json.dumps({"error": f"geocoding HTTP {last_status}", "city": city})


class WeatherAtLatLonTool:
    name = "weather_at_latlon"
    description = (
        "Retrieve 3 day hourly forecast from Open-Meteo for coordinates. "
        "Input may be {'lat': float, 'lon': float}, a JSON-string of that dict, or a string 'lat, lon'."
    )
    input_schema = {
        "type": "object",
        "properties": {"lat": {"type": "number"}, "lon": {"type": "number"}},
        "required": ["lat", "lon"]
    }

    BASE_URL = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude={lat}&longitude={lon}"
        "&hourly=temperature_2m,precipitation_probability,precipitation,"
        "wind_speed_10m,"
        "wind_gusts_10m,"
        "&forecast_days=3&wind_speed_unit=kn&temperature_unit=fahrenheit&precipitation_unit=inch"
    )

    def _parse_input(self, input_obj):
        # dict case
        if isinstance(input_obj, dict) and "lat" in input_obj and "lon" in input_obj:
            return float(input_obj["lat"]), float(input_obj["lon"])
        # JSON-string dict case
        if isinstance(input_obj, str):
            s = input_obj.strip()
            if s.startswith("{") and s.endswith("}"):
                try:
                    obj = json.loads(s)
                    if "lat" in obj and "lon" in obj:
                        return float(obj["lat"]), float(obj["lon"])
                except Exception:
                    pass
            # "lat, lon" case
            parts = s.replace(",", " ").split()
            nums = [p for p in parts if p.replace(".", "", 1).replace("-", "", 1).isdigit()]
            if len(nums) >= 2:
                return float(nums[0]), float(nums[1])
        # forgiving alternative keys
        if isinstance(input_obj, dict):
            for a, b in (("latitude", "longitude"), ("y", "x")):
                if a in input_obj and b in input_obj:
                    return float(input_obj[a]), float(input_obj[b])
        raise ValueError("weather_at_latlon input must be {'lat': ..., 'lon': ...}, a JSON-string of that dict, or 'lat, lon' string")

    def use(self, input_obj):
        lat, lon = self._parse_input(input_obj)
        lat = -90.0 if lat < -90.0 else 90.0 if lat > 90.0 else lat
        lon = ((lon + 180.0) % 360.0) - 180.0
        url = self.BASE_URL.format(lat=lat, lon=lon)

        with httpx.Client(timeout=12.0, headers={"User-Agent": "fairllm-agent-weather/1.0"}, http2=False) as cli:
            last_status = None
            for _ in range(3):
                r = cli.get(url)
                last_status = r.status_code
                if last_status == 200:
                    raw = r.json()
                    return json.dumps({
                        "source": "open-meteo",
                        "query": {"lat": lat, "lon": lon},
                        "hourly": raw.get("hourly", {}),
                        "meta": {
                            "elevation": raw.get("elevation"),
                            "timezone": raw.get("timezone"),
                            "tz_abbr": raw.get("timezone_abbreviation"),
                        }
                    })
                time.sleep(0.8)

        return json.dumps({"error": f"weather HTTP {last_status}", "lat": lat, "lon": lon})


class DroneProfileTool:
    """
    Normalize drone information into a structured limits object
    that other tools/agents can rely on.
    """
    name = "drone_profile"
    description = (
        "Given a drone model or explicit limits, return normalized safety limits. "
        "If a known model is provided, use a preset profile. "
        "Otherwise, accept explicit numeric limits. "
        "All wind limits are internally stored in knots to match weather data."
    )
    input_schema = {
        "type": "object",
        "properties": {
            "model": {"type": "string"},
            "max_wind_mph": {"type": "number"},
            "min_temp_f": {"type": "number"},
            "max_temp_f": {"type": "number"},
        },
        "required": [],
    }

    # Simple preset profiles for common/fictional drones
    PRESET_PROFILES = {
        # Example values; tweak if you like
        "dji mini 2":  {"max_wind_mph": 24, "min_temp_f": 14,  "max_temp_f": 104},
        "dji mavic 3": {"max_wind_mph": 26, "min_temp_f": 14,  "max_temp_f": 104},
        "dji avata":   {"max_wind_mph": 23, "min_temp_f": 32,  "max_temp_f": 104},
    }

    @staticmethod
    def _mph_to_knots(mph: float) -> float:
        # 1 knot ≈ 1.15078 mph
        return mph / 1.15078

    def _parse_input(self, input_obj):
        """
        Accept:
        - raw string: interpreted as model name
        - JSON-string: parsed to dict
        - dict: may contain model and/or explicit numeric limits
        """
        # Raw string: treat as model
        if isinstance(input_obj, str):
            s = input_obj.strip()
            if s.startswith("{") and s.endswith("}"):
                try:
                    return json.loads(s)
                except Exception:
                    pass
            return {"model": s}

        # Dict case
        if isinstance(input_obj, dict):
            return input_obj

        raise ValueError("drone_profile input must be a model name string or a dict of limits.")

    def use(self, input_obj):
        data = self._parse_input(input_obj)
        model = str(data.get("model") or "").strip()
        max_wind_mph = data.get("max_wind_mph")
        min_temp_f = data.get("min_temp_f")
        max_temp_f = data.get("max_temp_f")

        # If model matches a preset, fill from there (unless overridden explicitly)
        profile = {}
        if model:
            key = model.lower()
            if key in self.PRESET_PROFILES:
                profile.update(self.PRESET_PROFILES[key])
                profile["model"] = model

        # Override with any explicit numeric limits provided
        if max_wind_mph is not None:
            profile["max_wind_mph"] = float(max_wind_mph)
        if min_temp_f is not None:
            profile["min_temp_f"] = float(min_temp_f)
        if max_temp_f is not None:
            profile["max_temp_f"] = float(max_temp_f)

        # Basic sanity defaults if still missing
        if "max_wind_mph" not in profile:
            profile["max_wind_mph"] = 20.0  # conservative default
        if "min_temp_f" not in profile:
            profile["min_temp_f"] = 14.0
        if "max_temp_f" not in profile:
            profile["max_temp_f"] = 95.0

        # Convert mph → knots for internal use with Open-Meteo
        profile["max_wind_knots"] = self._mph_to_knots(profile["max_wind_mph"])

        # Mark that this is a normalized profile
        profile["source"] = "normalized_drone_profile"

        return json.dumps(profile)


class WeatherLimitCheckerTool:
    """
    Evaluate hourly weather against a drone profile and determine safe/unsafe hours.
    Rule: *any* nonzero precipitation makes that hour unsafe.
    """
    name = "weather_limit_checker"
    description = (
        "Given weather data (from weather_at_latlon) and a drone profile "
        "(from drone_profile), return per hour safety flags and contiguous safe windows. "
        "Any precipitation makes an hour unsafe."
    )
    input_schema = {
        "type": "object",
        "properties": {
            "weather": {"type": "object"},
            "drone_profile": {"type": "object"},
        },
        "required": ["weather", "drone_profile"],
    }

    def _parse_input(self, input_obj: Any) -> Dict[str, Any]:
        """
        Accepts:
        - dict with keys 'weather' and 'drone_profile'
        - JSON string encoding that dict
        """
        if isinstance(input_obj, dict):
            return input_obj
        if isinstance(input_obj, str):
            s = input_obj.strip()
            return json.loads(s)
        raise ValueError("weather_limit_checker input must be a dict or JSON string.")

    def use(self, input_obj: Any) -> str:
        try:
            payload = self._parse_input(input_obj)
        except Exception as e:
            return json.dumps({"error": "Invalid input to weather_limit_checker", "details": str(e)})

        weather: Dict[str, Any] = payload.get("weather", {})
        drone_profile: Dict[str, Any] = payload.get("drone_profile", {})

        hourly: Dict[str, List[Any]] = weather.get("hourly") or {}

        times: List[str] = hourly.get("time") or []
        temps_f: List[float] = hourly.get("temperature_2m") or []
        wind_kn: List[float] = hourly.get("wind_speed_10m") or []
        gust_kn: List[float] = hourly.get("wind_gusts_10m") or []
        precip_in: List[float] = hourly.get("precipitation") or []
        precip_prob: List[Optional[float]] = hourly.get("precipitation_probability") or []

        max_wind_knots: float = float(drone_profile.get("max_wind_knots", 0.0))
        min_temp_f: float = float(drone_profile.get("min_temp_f", -999.0))
        max_temp_f: float = float(drone_profile.get("max_temp_f", 999.0))

        hours_output: List[Dict[str, Any]] = []

        # Evaluate each hour
        for i, t in enumerate(times):
            temp = self._safe_get(temps_f, i)
            wind = self._safe_get(wind_kn, i)
            gust = self._safe_get(gust_kn, i)
            precip = self._safe_get(precip_in, i)
            prob = self._safe_get(precip_prob, i)

            is_safe = True
            reasons: List[str] = []

            # Temperature check
            if temp is not None:
                if temp < min_temp_f:
                    is_safe = False
                    reasons.append(f"Too cold ({temp:.1f} F < {min_temp_f:.1f} F)")
                if temp > max_temp_f:
                    is_safe = False
                    reasons.append(f"Too hot ({temp:.1f} F > {max_temp_f:.1f} F)")

            # Wind check (sustained)
            if wind is not None and max_wind_knots > 0.0:
                if wind > max_wind_knots:
                    is_safe = False
                    reasons.append(
                        f"Wind too strong ({wind:.1f} kt > {max_wind_knots:.1f} kt)"
                    )

            # Gust check (optional extra margin)
            if gust is not None and max_wind_knots > 0.0:
                # Example rule: gusts more than 5 kt above limit are unsafe
                if gust > max_wind_knots + 5.0:
                    is_safe = False
                    reasons.append(
                        f"Gusts too strong ({gust:.1f} kt > {max_wind_knots + 5.0:.1f} kt)"
                    )

            # Precipitation check: any precipitation at all is unsafe
            precip_val: Optional[float] = None
            if precip is not None:
                try:
                    precip_val = float(precip)
                except (TypeError, ValueError):
                    precip_val = None

            if precip_val is not None and precip_val > 0.0:
                is_safe = False
                reasons.append(
                    f"Precipitation present ({precip_val:.3f} in/hr; any precipitation not allowed)"
                )

            hours_output.append(
                {
                    "time": t,
                    "temperature_2m_f": temp,
                    "wind_speed_10m_kn": wind,
                    "wind_gusts_10m_kn": gust,
                    "precipitation_in": precip_val,
                    "precipitation_probability": prob,
                    "is_safe": is_safe,
                    "reasons": reasons,
                }
            )

        # Build continuous safe windows
        safe_windows: List[Dict[str, Any]] = []
        current_start: Optional[int] = None

        for idx, hour in enumerate(hours_output):
            if hour["is_safe"]:
                if current_start is None:
                    current_start = idx
            else:
                if current_start is not None:
                    safe_windows.append(self._build_window(hours_output, current_start, idx - 1))
                    current_start = None

        # Close window if it runs to the last hour
        if current_start is not None:
            safe_windows.append(self._build_window(hours_output, current_start, len(hours_output) - 1))

        total_hours = len(hours_output)
        num_safe_hours = sum(1 for h in hours_output if h["is_safe"])

        result = {
            "drone_profile": {
                "max_wind_knots": max_wind_knots,
                "min_temp_f": min_temp_f,
                "max_temp_f": max_temp_f,
            },
            "hours": hours_output,
            "safe_windows": safe_windows,
            "summary": {
                "total_hours": total_hours,
                "num_safe_hours": num_safe_hours,
                "num_safe_windows": len(safe_windows),
            },
        }

        return json.dumps(result)

    @staticmethod
    def _safe_get(lst: List[Any], idx: int) -> Any:
        if idx < 0 or idx >= len(lst):
            return None
        return lst[idx]

    def _build_window(self, hours: List[Dict[str, Any]], start_idx: int, end_idx: int) -> Dict[str, Any]:
        """Summarize a contiguous block of safe hours."""
        window_hours = hours[start_idx:end_idx + 1]
        times = [h["time"] for h in window_hours]
        winds = [h["wind_speed_10m_kn"] for h in window_hours if h["wind_speed_10m_kn"] is not None]
        gusts = [h["wind_gusts_10m_kn"] for h in window_hours if h["wind_gusts_10m_kn"] is not None]
        precips = [h["precipitation_in"] for h in window_hours if h["precipitation_in"] is not None]

        def _avg(xs):
            return sum(xs) / len(xs) if xs else None

        return {
            "start": times[0] if times else None,
            "end": times[-1] if times else None,
            "num_hours": len(window_hours),
            "avg_wind_knots": _avg(winds),
            "max_gust_knots": max(gusts) if gusts else None,
            "max_precip_in": max(precips) if precips else None,
        }


In [3]:
# ============================================================
# Cell 3 — Agent wiring (current weather agent, plus new tools registered)
# ============================================================

from fairlib import (
    ToolRegistry,
    ToolExecutor,
    WorkingMemory,
    SimpleAgent,
    SimpleReActPlanner,
    RoleDefinition,
)

# Register tools
tool_registry = ToolRegistry()
tool_registry.register_tool(GeocodeCityTool())
tool_registry.register_tool(WeatherAtLatLonTool())
tool_registry.register_tool(DroneProfileTool())
tool_registry.register_tool(WeatherLimitCheckerTool())

# Core agent components (this is still your weather Q&A agent)
executor = ToolExecutor(tool_registry)
memory = WorkingMemory()
planner = SimpleReActPlanner(llm, tool_registry)

# Strong role: force tool usage and require numeric outputs for the requested hour
planner.prompt_builder.role_definition = RoleDefinition(
    "You are a weather retrieval assistant. "
    "If the user provides a city name, first call the tool `geocode_city` to obtain lat and lon, "
    "then call the tool `weather_at_latlon` with those coordinates to fetch the forecast. "
    "If the user provides coordinates directly, call `weather_at_latlon` immediately. "
    "Do not invent numbers; always use the tools. "
    "Return exactly ONE JSON object as the final answer with keys: question, tool_calls, answer. "
    "In `answer`, include the specific hourly values for the time the user asked about. "
    "From the `weather_at_latlon` tool output, use `hourly.time` to locate the closest hour to the requested time "
    "(e.g., 'tomorrow at 2 PM'). If timezone is unclear, choose the closest matching hour string. "
    "Include, at minimum, these fields from the chosen hour: "
    "temperature_2m, relative_humidity_2m, wind_speed_10m, wind_direction_10m, wind_gusts_10m, "
    "precipitation_probability, precipitation. "
    "Also include `location` (lat, lon) and `chosen_time` (the ISO string from `hourly.time`) in `answer`."
)

# Optional: reinforce structured output
if hasattr(planner.prompt_builder, "output_format_hint"):
    planner.prompt_builder.output_format_hint = (
        "Final answer must be JSON with shape: "
        "{ 'question': str, "
        "  'tool_calls': [ { 'tool': str, 'input': object } ], "
        "  'answer': { "
        "     'location': { 'lat': number, 'lon': number }, "
        "     'chosen_time': str, "
        "     'values': { "
        "        'temperature_2m': number, "
        "        'relative_humidity_2m': number, "
        "        'wind_speed_10m': number, "
        "        'wind_direction_10m': number, "
        "        'wind_gusts_10m': number, "
        "        'precipitation_probability': number, "
        "        'precipitation': number "
        "     } "
        "  } "
        "}"
    )

# Assemble the current weather agent
agent = SimpleAgent(
    llm=llm,
    planner=planner,
    tool_executor=executor,
    memory=memory,
    max_steps=12,
)


Registering tool: geocode_city
Registering tool: weather_at_latlon
Registering tool: drone_profile
Registering tool: weather_limit_checker


In [4]:
# ============================================================
# Cell 6 — WeatherSafetyAgent (specialist for safe windows)
# ============================================================

# This agent is NOT the main user-facing agent.
# It is a specialist that, given a location and drone profile,
# fetches hourly weather and evaluates safe windows.

from fairlib import SimpleAgent, SimpleReActPlanner, WorkingMemory, RoleDefinition

# Separate planner + memory for the safety agent (reuse same tool_registry and llm)
safety_memory = WorkingMemory()
safety_planner = SimpleReActPlanner(llm, tool_registry)

safety_planner.prompt_builder.role_definition = RoleDefinition(
    "You are a drone weather safety analysis assistant. "
    "Your input will describe a specific location and a drone profile. "
    "Your job is to fetch the hourly weather for that location using the tool `weather_at_latlon`, "
    "then evaluate which hours are safe for that drone using the tool `weather_limit_checker`. "
    "You MUST call `weather_at_latlon` first to obtain the forecast. "
    "Then you MUST call `weather_limit_checker` exactly once, passing it a JSON object with keys "
    "`weather` (the full JSON returned by weather_at_latlon) and `drone_profile` (the drone limits object). "
    "Do not invent weather values; always rely on the tools. "
    "You are NOT talking directly to an end-user; you are producing structured data for another agent. "
    "Your final answer MUST be exactly ONE JSON object with keys: "
    "  'location'   : information about the location (at least lat, lon, and optionally a name), "
    "  'drone_profile' : the limits used (max_wind_knots, min_temp_f, max_temp_f), "
    "  'hours'      : the list of hourly records with safety flags as returned by weather_limit_checker, "
    "  'safe_windows' : the list of safe time windows as returned by weather_limit_checker, "
    "  'summary'    : the summary object from weather_limit_checker. "
    "If the input includes a 'location' object, propagate its name or city field into the final 'location' field."
)

# Optional: give a format hint to keep things clean
if hasattr(safety_planner.prompt_builder, "output_format_hint"):
    safety_planner.prompt_builder.output_format_hint = (
        "Final answer must be JSON with shape: "
        "{ "
        "  'location': { 'lat': number, 'lon': number, 'name': string }, "
        "  'drone_profile': { "
        "      'max_wind_knots': number, "
        "      'min_temp_f': number, "
        "      'max_temp_f': number "
        "  }, "
        "  'hours': [ "
        "      { "
        "        'time': string, "
        "        'temperature_2m_f': number, "
        "        'wind_speed_10m_kn': number, "
        "        'wind_gusts_10m_kn': number, "
        "        'precipitation_in': number, "
        "        'precipitation_probability': number, "
        "        'is_safe': boolean "
        "      } "
        "  ], "
        "  'safe_windows': [ "
        "      { "
        "        'start': string, "
        "        'end': string, "
        "        'num_hours': number, "
        "        'avg_wind_knots': number, "
        "        'max_gust_knots': number, "
        "        'max_precip_in': number "
        "      } "
        "  ], "
        "  'summary': { "
        "      'total_hours': number, "
        "      'num_safe_hours': number, "
        "      'num_safe_windows': number "
        "  } "
        "}"
    )

weather_safety_agent = SimpleAgent(
    llm=llm,
    planner=safety_planner,
    tool_executor=executor,   # reuse the same ToolExecutor, which has weather_limit_checker etc.
    memory=safety_memory,
    max_steps=12,
)


In [5]:
# ============================================================
# Cell 7 — CoordinatorAgent (user-facing drone flight planner)
# ============================================================

from fairlib import SimpleAgent, SimpleReActPlanner, WorkingMemory, RoleDefinition

# Separate planner + memory for the coordinator (top-level) agent
coord_memory = WorkingMemory()
coord_planner = SimpleReActPlanner(llm, tool_registry)

coord_planner.prompt_builder.role_definition = RoleDefinition(
    "You are a drone flight safety assistant. "
    "Your goal is to determine whether it is safe to fly a given drone in a specified city "
    "at a requested time, based on the drone's safety limits and the weather forecast. "
    "You interact directly with the user.\n\n"
    "Tool usage protocol:\n"
    "1) Parse the user's message to identify:\n"
    "   - the city (for example, 'Denver'),\n"
    "   - the drone model (for example, 'DJI Mini 2'),\n"
    "   - the desired time window (for example, 'tomorrow afternoon').\n"
    "2) Call the `drone_profile` tool with the model string to obtain normalized limits.\n"
    "3) Call the `geocode_city` tool with the city string to obtain latitude and longitude.\n"
    "4) Call the `weather_at_latlon` tool once, using the coordinates from step (3) and a time range\n"
    "   that covers the requested window (for example, the next 24–48 hours).\n"
    "5) Call the `weather_limit_checker` tool exactly once, passing a JSON object with keys\n"
    "   `weather` and `drone_profile` that come directly from tools (no hand-written fields).\n"
    "6) Use the result of `weather_limit_checker` to decide whether flying is safe in that city\n"
    "   during the requested time window. If unsafe, identify the best alternative safe window\n"
    "   from the returned safe windows.\n\n"
    "Output protocol:\n"
    "- After finishing all reasoning and tool calls, you MUST write a line that contains exactly:\n"
    "  Final Answer:\n"
    "- On the next line, output a single JSON object with keys `query`, `drone_profile`,\n"
    "  `safety`, `recommendations`, and `notes`.\n"
    "- `query` should be a JSON object describing the interpreted city, drone model, and time window.\n"
    "- `drone_profile` must be exactly the JSON returned by the `drone_profile` tool.\n"
    "- `safety` must be exactly the JSON returned by the `weather_limit_checker` tool\n"
    "  (with its `hours`, `safe_windows`, and `summary` fields).\n"
    "- `recommendations` should contain fields like `is_safe_now`, `best_window`, and `reason`.\n"
    "- `notes` should be a concise natural-language explanation for the pilot.\n"
    "Do NOT call any tool that looks up individual parks or named locations, and do NOT mention\n"
    "specific parks in the final JSON. You are evaluating safety for the city as a whole."
)


# Optional: enforce the JSON shape more explicitly
if hasattr(coord_planner.prompt_builder, "output_format_hint"):
    coord_planner.prompt_builder.output_format_hint = (
        "After the line 'Final Answer:', you MUST output a single JSON object with keys "
        "'query', 'drone_profile', 'safety', 'recommendations', and 'notes'."
    )


coordinator_agent = SimpleAgent(
    llm=llm,
    planner=coord_planner,
    tool_executor=executor,   # same ToolExecutor, with all tools registered
    memory=coord_memory,
    max_steps=12,
)


In [6]:
# Cell 4 — Helper to run a prompt and pretty print JSON
def pretty_print(json_text: str):
    try:
        print(json.dumps(json.loads(json_text), indent=2))
    except Exception:
        print(json_text)

async def ask(q: str):
    print("USER:", q)
    resp = await coordinator_agent.arun(q)
    pretty_print(resp)


In [7]:
# Cell 5 — Request
await ask("I want to fly my DJI Mini 2 in Fairbanks today.")


USER: I want to fly my DJI Mini 2 in Fairbanks today.
--- Step 1/12 ---
Thought: The user wants to fly a DJI Mini 2 drone in Fairbanks today. I need to check the drone profile for DJI Mini 2, geocode Fairbanks to get its coordinates, get the weather forecast for today in Fairbanks, and then check if the weather conditions are safe for flying the DJI Mini 2 today.
Action: Using tool 'drone_profile' with input 'DJI Mini 2'
Observation: {"max_wind_mph": 24, "min_temp_f": 14, "max_temp_f": 104, "model": "DJI Mini 2", "max_wind_knots": 20.855419802221103, "source": "normalized_drone_profile"}
--- Step 2/12 ---
Thought: I have the drone profile for the DJI Mini 2. The next step is to geocode the city "Fairbanks" to get its latitude and longitude for weather lookup.
Action: Using tool 'geocode_city' with input 'Fairbanks'
Observation: {"city": "Fairbanks", "country": "United States", "lat": 64.83778, "lon": -147.71638}
--- Step 3/12 ---
Thought: I have the drone profile and the coordinates fo